# Guardrailer — GPU Embedding Generation & Qdrant Ingestion

This notebook generates 1024-dim embeddings for the Guardrailer security dataset using a T4 GPU,
with fault-tolerant checkpointing and 70MB chunked output for local Qdrant ingestion.

## Pipeline
1. **Upload** `enhanced_payloads.parquet` to Kaggle
2. **Generate embeddings** with `BAAI/bge-large-en-v1.5` on T4 GPU
3. **Checkpoint** after every batch — resume on kernel restart
4. **Chunk** embeddings into 70MB `.npy` files
5. **Download** chunks + payloads for local ingestion

## 0. Setup & GPU Check

In [ ]:
import os, sys, json, time, gc, shutil, signal, atexit
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# GPU check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"GPU count: {num_gpus}")
    for i in range(num_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} -- {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Embeddings will be slow on CPU.")

# Disk space check
disk = shutil.disk_usage("/kaggle/working")
free_gb = disk.free / 1e9
print(f"\nDisk free: {free_gb:.1f} GB")
if free_gb < 20:
    print(f"  WARNING: Low disk space. Need ~10GB for embeddings + zip.")

# Paths
WORK_DIR = Path("/kaggle/working/guardrailer")
DATA_DIR = WORK_DIR / "data"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
CHUNK_DIR = WORK_DIR / "embedding_chunks"
OUTPUT_DIR = WORK_DIR / "output"

for d in [WORK_DIR, DATA_DIR, CHECKPOINT_DIR, CHUNK_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nWork directory: {WORK_DIR}")

## 1. Load Dataset

Upload `enhanced_payloads.parquet` to Kaggle Datasets or place it in `/kaggle/input/`.

In [ ]:
# Try to find the parquet file
possible_paths = [
    Path("/kaggle/input/datasets/prashannadeveloper/guardrailer-dataset-v1/guardrailer_dataset_v1.parquet"),
    Path("/kaggle/working/enhanced_payloads.parquet"),
    DATA_DIR / "enhanced_payloads.parquet",
]

parquet_path = None
for p in possible_paths:
    if p.exists():
        parquet_path = p
        break

if parquet_path is None:
    print("ERROR: enhanced_payloads.parquet not found.")
    print("Upload it to Kaggle Datasets or place in /kaggle/working/")
    print("Expected columns: id, prompt_text, is_malicious, attack_category, ...")
else:
    df = pd.read_parquet(parquet_path)
    print(f"Loaded: {len(df):,} rows from {parquet_path.name}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nCategory distribution:")
    for cat in sorted(df["attack_category"].unique()):
        n = (df["attack_category"] == cat).sum()
        print(f"  {cat:30s}: {n:>8,}")

## 2. Embedding Generation with Checkpointing

**Dual-T4 GPU Optimization (30GB total VRAM):**
- Multi-process pool — each GPU gets its own model copy for true parallelism
- `batch_size=512` — auto-scales to 256 per GPU
- `max_length=128` — covers 95%+ of texts
- `tqdm` progress bar with ETA and speed
- Checkpoint after every batch for fault tolerance
- OOM recovery: halves batch_size and retries


In [ ]:
import signal
import atexit

class EmbeddingGenerator:
    """
    Multi-GPU embedding generator with fault-tolerant checkpointing.
    Uses SentenceTransformer multi-process pool for true multi-GPU parallelism.
    """

    def __init__(self, model_name="BAAI/bge-large-en-v1.5", batch_size=512, max_length=128):
        self.model_name = model_name
        self.batch_size = batch_size
        self.max_length = max_length
        self.model = None
        self.device = None
        self.num_gpus = 0
        self.pool = None

    def load_model(self):
        """Load sentence-transformers model with multi-process pool for multi-GPU."""
        print(f"Loading {self.model_name}...")
        self.model = SentenceTransformer(self.model_name)

        self.num_gpus = torch.cuda.device_count()
        print(f"  Detected {self.num_gpus} GPU(s)")

        if self.num_gpus >= 2:
            self.device = torch.device("cuda:0")
            print(f"  Starting multi-process pool across {self.num_gpus} GPUs...")
            self.pool = self.model.start_multi_process_pool(
                [f"cuda:{i}" for i in range(self.num_gpus)]
            )
            self.batch_size = max(self.batch_size, 256 * self.num_gpus)
            print(f"  Batch size: {self.batch_size}")
        elif self.num_gpus == 1:
            self.device = torch.device("cuda:0")
            self.model = self.model.to("cuda:0")
            print(f"  Model on GPU: {torch.cuda.get_device_name(0)}")
        else:
            self.device = torch.device("cpu")
            print(f"  Model on CPU (slow)")

        self.embed_dim = self.model.get_sentence_embedding_dimension()
        print(f"  Embedding dim: {self.embed_dim}")

    def encode_batch_safe(self, texts, batch_size):
        """Encode with OOM handling — returns None on OOM."""
        try:
            encode_kwargs = dict(
                batch_size=batch_size,
                show_progress_bar=False,
                normalize_embeddings=True,
                convert_to_numpy=True,
            )
            if self.pool is not None:
                embeddings = self.model.encode(texts, pool=self.pool, **encode_kwargs)
            else:
                with torch.no_grad():
                    torch.cuda.synchronize()
                    embeddings = self.model.encode(texts, **encode_kwargs)
            return embeddings.astype(np.float16)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            return None

    def _save_checkpoint(self, embeddings_file, progress_file, all_embeddings, last_idx, total,
                         batch_num, current_batch_size, oom_retries, completed=False):
        """Save checkpoint to disk."""
        np.save(embeddings_file, all_embeddings[:last_idx])
        with open(progress_file, "w") as f:
            json.dump({
                "last_idx": last_idx, "total": total,
                "batch_num": batch_num, "batch_size": current_batch_size,
                "oom_retries": oom_retries, "completed": completed,
                "timestamp": datetime.now().isoformat(),
            }, f)

    def generate(self, texts, checkpoint_path, resume=True):
        """
        Generate embeddings with multi-GPU parallelism and tqdm progress.
        Saves checkpoint after every batch. Handles interrupts gracefully.
        """
        from tqdm.notebook import tqdm

        total = len(texts)
        embeddings_file = checkpoint_path.with_suffix(".npy")
        progress_file = checkpoint_path.with_suffix(".json")

        last_idx = 0
        all_embeddings = None
        current_batch_size = self.batch_size

        if resume and embeddings_file.exists() and progress_file.exists():
            with open(progress_file) as f:
                progress = json.load(f)
            last_idx = progress.get("last_idx", 0)
            current_batch_size = progress.get("batch_size", self.batch_size)
            print(f"  Checkpoint found: last_idx={last_idx:,}, batch_size={current_batch_size}")

            if last_idx > 0:
                all_embeddings = np.load(embeddings_file)
                file_rows = all_embeddings.shape[0]
                print(f"  NPY file has {file_rows:,} rows, expected {last_idx:,}")

                if file_rows != last_idx:
                    print(f"  MISMATCH! Backing up corrupt checkpoint...")
                    backup_npy = embeddings_file.with_suffix(".corrupt.npy")
                    backup_json = progress_file.with_suffix(".corrupt.json")
                    embeddings_file.rename(backup_npy)
                    progress_file.rename(backup_json)
                    print(f"  Backed up to {backup_npy.name} / {backup_json.name}")
                    all_embeddings = None
                    last_idx = 0
                else:
                    print(f"  Resuming from {last_idx:,}/{total:,} (batch_size={current_batch_size})")

        if all_embeddings is None:
            all_embeddings = np.zeros((total, self.embed_dim), dtype=np.float16)
            last_idx = 0

        start_time = time.time()
        oom_retries = 0
        max_oom_retries = 5
        batch_num = last_idx // current_batch_size
        last_saved_idx = last_idx

        interrupted = False
        try:
            pbar = tqdm(total=total, initial=last_idx, desc="Embedding",
                        unit=" texts", ncols=100, leave=True)

            idx = last_idx
            while idx < total:
                batch_end = min(idx + current_batch_size, total)
                batch_texts = texts[idx:batch_end]

                batch_emb = None
                attempts = 0

                while batch_emb is None and attempts < max_oom_retries:
                    batch_emb = self.encode_batch_safe(batch_texts, current_batch_size)

                    if batch_emb is None:
                        oom_retries += 1
                        old_bs = current_batch_size
                        current_batch_size = max(64, current_batch_size // 2)
                        print(f"\n  OOM! {old_bs} -> {current_batch_size} (retry {attempts+1}/{max_oom_retries})")
                        attempts += 1

                if batch_emb is None:
                    print(f"\n  FATAL OOM after {max_oom_retries} retries")
                    self._save_checkpoint(embeddings_file, progress_file, all_embeddings,
                                          last_saved_idx, total, batch_num, current_batch_size, oom_retries)
                    pbar.close()
                    return all_embeddings

                all_embeddings[idx:batch_end] = batch_emb
                last_saved_idx = batch_end
                batch_num += 1

                self._save_checkpoint(embeddings_file, progress_file, all_embeddings,
                                      last_saved_idx, total, batch_num, current_batch_size, oom_retries)

                elapsed = time.time() - start_time
                processed = last_saved_idx
                rate = processed / elapsed if elapsed > 0 else 0
                eta = (total - last_saved_idx) / rate if rate > 0 else 0

                pbar.update(batch_end - pbar.n)
                pbar.set_postfix({
                    "bs": current_batch_size,
                    "gpus": self.num_gpus,
                    "speed": f"{rate:.0f}/s",
                    "ETA": f"{eta/60:.1f}m"
                })

                if batch_num % 100 == 0:
                    torch.cuda.empty_cache()

                idx = batch_end

            pbar.close()

        except KeyboardInterrupt:
            interrupted = True
            print(f"\n  Interrupted! Saving checkpoint at {last_saved_idx:,}/{total:,}...")

        self._save_checkpoint(embeddings_file, progress_file, all_embeddings,
                              last_saved_idx, total, batch_num, current_batch_size, oom_retries,
                              completed=(not interrupted and last_saved_idx >= total))

        elapsed = time.time() - start_time
        status = "completed" if not interrupted else "interrupted"
        print(f"\n  {status}: {last_saved_idx:,}/{total:,} embeddings in {elapsed:.1f}s ({last_saved_idx/elapsed:.0f} texts/s)")
        print(f"  Shape: {all_embeddings.shape}, GPUs: {self.num_gpus}, OOM retries: {oom_retries}")

        return all_embeddings


print("Multi-GPU EmbeddingGenerator defined.")


In [ ]:
# Initialize generator
generator = EmbeddingGenerator(
    model_name="BAAI/bge-large-en-v1.5",
    batch_size=512,
    max_length=128,
)

# Load model
generator.load_model()

In [ ]:
# Extract texts
texts = df["prompt_text"].astype(str).tolist()
print(f"Embedding {len(texts):,} texts...")

# Generate embeddings with checkpointing
checkpoint_path = CHECKPOINT_DIR / "embeddings"
embeddings = generator.generate(texts, checkpoint_path, resume=True)

print(f"\nFinal embeddings shape: {embeddings.shape}")
print(f"Memory: {embeddings.nbytes / 1e6:.1f} MB")

## 3. Chunk Embeddings into 70MB Files

In [ ]:
def chunk_embeddings(embeddings, chunk_size_mb=70, output_dir=CHUNK_DIR):
    """
    Partition embeddings into chunks of exactly chunk_size_mb each.

    Each chunk is saved as a .npy file with metadata in a companion .json.
    """
    chunk_size_bytes = chunk_size_mb * 1024 * 1024
    bytes_per_row = embeddings.dtype.itemsize * embeddings.shape[1]
    rows_per_chunk = chunk_size_bytes // bytes_per_row

    total_rows = embeddings.shape[0]
    total_chunks = int(np.ceil(total_rows / rows_per_chunk))

    print(f"Embedding shape: {embeddings.shape}")
    print(f"Dtype: {embeddings.dtype} ({embeddings.dtype.itemsize} bytes/element)")
    print(f"Bytes per row: {bytes_per_row}")
    print(f"Rows per chunk: {rows_per_chunk:,}")
    print(f"Total chunks: {total_chunks}")
    print(f"Target chunk size: {chunk_size_mb} MB")

    chunk_files = []
    for i in range(total_chunks):
        start = i * rows_per_chunk
        end = min(start + rows_per_chunk, total_rows)
        chunk = embeddings[start:end]

        chunk_path = output_dir / f"embeddings_chunk_{i:04d}.npy"
        np.save(chunk_path, chunk)

        actual_size = chunk_path.stat().st_size / 1e6

        # Save metadata
        meta_path = output_dir / f"embeddings_chunk_{i:04d}.json"
        meta = {
            "chunk_index": i,
            "total_chunks": total_chunks,
            "start_idx": start,
            "end_idx": end,
            "rows": end - start,
            "shape": list(chunk.shape),
            "dtype": str(chunk.dtype),
            "size_mb": round(actual_size, 2),
        }
        with open(meta_path, "w") as f:
            json.dump(meta, f, indent=2)

        chunk_files.append((chunk_path, actual_size))
        print(f"  Chunk {i:04d}: rows {start:,}-{end:,} | {actual_size:.1f} MB")

    print(f"\n✓ {total_chunks} chunks saved to {output_dir}")
    return chunk_files


# Chunk the embeddings
chunk_files = chunk_embeddings(embeddings, chunk_size_mb=70)

# Clean up large embedding array from memory
del embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Prepare Payloads for Download

In [ ]:
# Save enhanced payloads (already correct schema)
payloads_path = OUTPUT_DIR / "enhanced_payloads.parquet"
df.to_parquet(payloads_path, index=False)
print(f"Saved payloads: {payloads_path} ({payloads_path.stat().st_size / 1e6:.1f} MB)")

# Copy checkpoint for reference
checkpoint_path = CHECKPOINT_DIR / "embeddings.json"
if checkpoint_path.exists():
    shutil.copy2(checkpoint_path, OUTPUT_DIR / "embedding_checkpoint.json")

# List all output files
print(f"\nOutput files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")
for f in sorted(CHUNK_DIR.glob("*.npy")):
    print(f"  chunks/{f.name}: {f.stat().st_size / 1e6:.1f} MB")

## 5. Build corpus_meta.json

In [ ]:
import math

SPARSE_KEYWORDS = [
    "ignore previous", "override", "bypass", "jailbreak", "system prompt",
    "your instructions", "forget", "disregard", "dan", "do anything now",
    "act as", "roleplay", "pretend you", "hypothetical", "in theory",
    "markdown injection", "code comment", "readme", "yaml", "json payload",
    "<script>", "]]>", "```", "<!--", "-->", "eval(", "exec(",
    "base64", "rot13", "hex encoded", "obfuscated",
    "ignore all", "new instructions", "you are now", "persona",
    "reveal", "output", "display", "print", "show", "expose",
    "previous instructions", "earlier instructions", "initial instructions",
    "developer mode", "debug mode", "admin mode", "root mode",
    "you must", "you will", "you shall", "comply", "obey",
    "no restrictions", "no rules", "no limits", "unrestricted",
    "evil", "uncensored", "unfiltered", "without guidelines",
]

def build_corpus_meta(df, embeddings_file_or_array):
    """Build corpus metadata for scoring module."""
    if isinstance(embeddings_file_or_array, (str, Path)):
        # Load all chunks and concatenate
        chunks = []
        chunk_dir = embeddings_file_or_array
        for f in sorted(Path(chunk_dir).glob("*.npy")):
            chunks.append(np.load(f))
        embeddings = np.vstack(chunks)
    else:
        embeddings = embeddings_file_or_array

    texts = df["prompt_text"].astype(str).str.lower().tolist()
    N = len(texts)

    # IDF
    doc_freq = {}
    total_words = 0
    for text in texts:
        words = text.split()
        total_words += len(words)
        for w in set(words):
            doc_freq[w] = doc_freq.get(w, 0) + 1

    avgdl = total_words / N if N > 0 else 100.0

    keyword_idf = {}
    for kw in SPARSE_KEYWORDS:
        df_count = sum(1 for t in texts if kw in t)
        if df_count > 0:
            keyword_idf[kw] = math.log((N - df_count + 0.5) / (df_count + 0.5) + 1.0)
        else:
            keyword_idf[kw] = 1.0

    # Category centroids
    category_centroids = {}
    for cat in df["attack_category"].unique():
        mask = df["attack_category"] == cat
        if mask.sum() > 0:
            cat_emb = embeddings[mask.values].astype(np.float32)
            centroid = cat_emb.mean(axis=0)
            centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
            category_centroids[cat] = centroid.tolist()

    scoring_weights = {
        "dense": 0.30, "sparse_idf": 0.18, "centroid": 0.12,
        "cross_encoder": 0.12, "perplexity": 0.05, "entropy": 0.05,
        "token_frequency": 0.03, "ngram_overlap": 0.02,
        "uniqueness": 0.05, "length_norm": 0.05, "ensemble_bonus": 0.03,
    }

    meta = {
        "keyword_idf": keyword_idf,
        "total_documents": N,
        "avg_doc_length": avgdl,
        "avg_text_length": avgdl,
        "category_centroids": category_centroids,
        "cluster_centers": {},
        "scoring_weights": scoring_weights,
    }

    print(f"Built corpus_meta: {N:,} docs, {len(keyword_idf)} IDF keywords, {len(category_centroids)} centroids")
    return meta


# Build corpus_meta using chunked embeddings
meta = build_corpus_meta(df, CHUNK_DIR)

meta_path = OUTPUT_DIR / "corpus_meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Saved: {meta_path} ({meta_path.stat().st_size / 1e6:.1f} MB)")

## 6. Download All Files

Download the output directory to your local machine for Qdrant ingestion.

In [ ]:
import zipfile
from pathlib import Path

zip_path = WORK_DIR / "guardrailer_embeddings.zip"

# Auto-discover all files under WORK_DIR
all_files = sorted(WORK_DIR.rglob("*"))
data_files = [f for f in all_files if f.is_file()]

if not data_files:
    print("No files found. Run the embedding generation first.")
else:
    print(f"Found {len(data_files)} files to zip:")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in data_files:
            arcname = f.relative_to(WORK_DIR)
            zf.write(f, arcname)
            size_mb = f.stat().st_size / 1e6
            print(f"  {str(arcname):60s} {size_mb:8.1f} MB")

    total_mb = zip_path.stat().st_size / 1e6
    print(f"\nZip: {zip_path.name} ({total_mb:.1f} MB)")

    # Print zip contents summary
    with zipfile.ZipFile(zip_path, "r") as zf:
        by_dir = {}
        for info in zf.infolist():
            parts = info.filename.split("/")
            folder = parts[0] if len(parts) > 1 else "(root)"
            by_dir.setdefault(folder, []).append(info)

        print("\nZip structure:")
        for folder, files in by_dir.items():
            total = sum(f.file_size for f in files) / 1e6
            print(f"  {folder}/ ({len(files)} files, {total:.1f} MB)")
            for f in files[:5]:
                print(f"    {f.filename}")
            if len(files) > 5:
                print(f"    ... +{len(files)-5} more")

    print(f"\nDownload from Kaggle Output panel -> {zip_path.name}")